<a href="https://colab.research.google.com/github/sakshumvij/GNN_Stroke_Outcome_Prediction/blob/main/GNN_Stroke_Outcome_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **GNN Stroke Outcome Prediction Code**

Paper Title: Analyzing Structural Brain Connectivity with Graph Neural Networks to Predict Hemorrhagic Stroke Recovery


This notebook implements and evaluates a graph neural network (GNN) framework for predicting functional outcome following hemorrhagic stroke using structural brain connectivity. Four models are trained and compared: a multimodal GNN combining structural connectivity with clinical features, a graph-only GNN, a LASSO connectivity model, and a clinical baseline logistic regression. A post-hoc edge importance explainer identifies the connectivity patterns most predictive of poor outcome.

# Dependencies and Set Up

In [ ]:
# Install Dependencies

!pip install "tensorflow-gnn>=1.0.0" -q
!pip install nilearn -q

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

This notebook requires **TensorFlow 2.x** with the TF-GNN extension for graph neural network operations. The **TF_USE_LEGACY_KERAS** flag must be set before import to ensure compatibility between TF-GNN and the Keras API. Run this cell first to confirm correct library versions are installed before executing any model cells.

In [ ]:
import tensorflow as tf
import tensorflow_gnn as tfgnn

print("TensorFlow version:", tf.__version__)
print("TF-GNN version:", tfgnn.__version__)

TensorFlow version: 2.20.0
TF-GNN version: 1.0.3


# Multimodal GNN
Weighted GraphSAGE network that fuses structural connectivity graphs with clinical features via a gated joint embedding, trained with focal loss to handle class imbalance.

In [ ]:
# Main Multimodal Model

# Imports
import numpy as np
import os
import gc
from scipy.io import loadmat
import pandas as pd
import tensorflow as tf
import tensorflow_gnn as tfgnn
from tensorflow_gnn import GraphTensorSpec
from scipy import stats
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")

# Reproducibility
SEED = 21
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Hyperparameters
HIDDEN_DIM = 48
NUM_LAYERS = 3
NODE_FEAT_DIM = 5 # degree, strength, clustering, pagerank, participation coefficient
DROPOUT_RATE = 0.45
L2_REG = 8e-4
BATCH_SIZE = 16
EPOCHS = 400
PATIENCE = 60
BOOTSTRAP_N = 100000  # High resampling count ensures stable, highly precise 95% CI bounds (can be reduced for runtine concerns)
LR_INIT = 5e-4
CLINICAL_DIM = 2 # ICH Volume and ICH Location

# Data Split
TRAIN_FRAC = 0.60
VAL_FRAC = 0.15
TEST_FRAC = 0.25

# Path Configurations
DATA_DIR = "./data"
CLINICAL_CSV_PATH = os.path.join(DATA_DIR, "CLINICAL_CSV_PATH.csv")
ROSE_LABELS_CSV_PATH = os.path.join(DATA_DIR, "CSV_PATH.csv")
CONNECTIVITY_MAT_DIR = os.path.join(DATA_DIR, "MAT_DIR")

# Output Paths
OUTPUT_PLOT_PATH = "./outputs/gnn_multimodal_results.png"
PERMANENT_WEIGHTS_PATH = "./outputs/multimodel_gnn_weights.weights.h5"
CKPT_PATH = "/tmp/best_gnn_v6.weights.h5"


# Focal Loss
# Selected over standard Binary Cross-Entropy to mitigate class imbalance
# in mRS distributions. It down-weights well-classified examples to
# force the network to focus on the difficult clinical cases.

def focal_loss(gamma=2.0, alpha=0.25, label_smoothing=0.05):
    bce = tf.keras.losses.BinaryCrossentropy(
        label_smoothing=label_smoothing,
        reduction=tf.keras.losses.Reduction.NONE,
    )
    def loss_fn(y_true, y_pred):
        bce_val = bce(tf.expand_dims(y_true, -1), tf.expand_dims(y_pred, -1))
        p_t = tf.where(tf.equal(y_true, 1.0), y_pred, 1.0 - y_pred)
        fl = alpha * tf.pow(1.0 - p_t, gamma) * bce_val
        return tf.reduce_mean(fl)
    return loss_fn


# Smoothed Early Stopping
# Used instead of standard early stopping to handle validation volatility
# inherent in smaller clinical neuroimaging datasets.

class SmoothedEarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, monitor="val_auc", patience=PATIENCE, window=5):
        super().__init__()
        self.monitor = monitor
        self.patience = patience
        self.window = window
        self._history = []
        self._wait = 0
        self._best = -np.inf
        self._best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        val = (logs or {}).get(self.monitor)
        if val is None:
            return
        self._history.append(val)
        smoothed = np.mean(self._history[-self.window:])
        if smoothed > self._best:
            self._best = smoothed
            self._wait = 0
            self._best_weights = self.model.get_weights()
        else:
            self._wait += 1
            if self._wait >= self.patience:
                self.model.stop_training = True
                if self._best_weights:
                    self.model.set_weights(self._best_weights)

    def on_train_end(self, logs=None):
        if self._best_weights:
            self.model.set_weights(self._best_weights)


# Validation and Train Gap Monitoring
# Learning rate scheduler that dampens learning rate if the model
# begins to overfit on training samples in comparison to validation
class ValTrainGapMonitor(tf.keras.callbacks.Callback):
    def __init__(self, gap_threshold=0.12, cooldown=15, factor=0.5):
        super().__init__()
        self.gap_threshold = gap_threshold
        self.cooldown = cooldown
        self.factor = factor
        self._cooldown_cnt = 0
        self._train_ema = None
        self._ema_alpha = 0.15

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        train_auc = logs.get("auc", None)
        val_auc = logs.get("val_auc", None)
        if train_auc is None or val_auc is None:
            return
        self._train_ema = (
            train_auc if self._train_ema is None
            else self._ema_alpha * train_auc + (1 - self._ema_alpha) * self._train_ema
        )
        if self._cooldown_cnt > 0:
            self._cooldown_cnt -= 1
            return
        if val_auc - self._train_ema < -self.gap_threshold:
            new_lr = max(float(self.model.optimizer.learning_rate) * self.factor, 1e-6)
            tf.keras.backend.set_value(self.model.optimizer.learning_rate, new_lr)
            self._cooldown_cnt = self.cooldown


# Bootstrapped 95% CI

def bootstrap_auc_ci(y_true, y_pred, n=BOOTSTRAP_N, seed=SEED):

    # Returns (point_estimate, ci_lower, ci_upper) using percentile bootstrap

    rng = np.random.default_rng(seed)
    n_ = len(y_true)
    aucs = []
    for _ in range(n):
        idx = rng.integers(0, n_, size=n_)
        yt, yp = y_true[idx], y_pred[idx]
        if len(np.unique(yt)) < 2:
            continue
        aucs.append(roc_auc_score(yt, yp))
    aucs = np.array(aucs)
    pt = roc_auc_score(y_true, y_pred)
    return pt, np.percentile(aucs, 2.5), np.percentile(aucs, 97.5), aucs


def bootstrap_metric_ci(y_true, y_pred_binary, metric_fn, n=BOOTSTRAP_N, seed=SEED):

    # Bootstrap CI for scalar metrics (accuracy, F1, etc.)

    rng = np.random.default_rng(seed)
    n_ = len(y_true)
    vals = []
    for _ in range(n):
        idx = rng.integers(0, n_, size=n_)
        vals.append(metric_fn(y_true[idx], y_pred_binary[idx]))
    vals = np.array(vals)
    pt = metric_fn(y_true, y_pred_binary)
    return pt, np.percentile(vals, 2.5), np.percentile(vals, 97.5)


# Load Clinical Data

clinical_df = pd.read_csv(CLINICAL_CSV_PATH)
clinical_df["ID_int"] = (
    clinical_df["ID"].astype(str).str.replace("-", "").astype(int) # Aligns string patient IDs in CSV with numerical matrix filenames
)
clinical_encoded = pd.get_dummies(
    clinical_df, columns=["ICH_Location"], drop_first=True, dtype=float
)
CLINICAL_FEATURES = ["ICH_Volume", "ICH_Location_2"]
print(f"Clinical features ({CLINICAL_DIM}): {CLINICAL_FEATURES}")

clinical_lookup_raw = {}
for _, row in clinical_encoded.iterrows():
    sid = int(row["ID_int"])
    vals = [float(row[f]) if f in row.index else np.nan for f in CLINICAL_FEATURES]
    clinical_lookup_raw[sid] = np.array(vals, dtype=np.float32)


# Node Features

def node_features(connectivity: np.ndarray) -> np.ndarray:
    W = np.abs(connectivity).astype(np.float64)
    n = W.shape[0]

    A = (W > 0).astype(np.float64)
    degree = A.sum(axis=1)
    strength = W.sum(axis=1)

    # Vectorized clustering coefficient via matrix trace of the cube-rooted weights
    W_cbrt = np.cbrt(W)
    triangles = (W_cbrt @ W_cbrt * W_cbrt).diagonal()
    ki = degree
    denom = ki * (ki - 1)
    clustering = np.where(denom > 0, triangles / denom, 0.0)

    # Power iteration implementation of PageRank centrality (10 iterations)
    d = 0.85
    col_sums = W.sum(axis=0)
    col_sums[col_sums == 0] = 1.0
    T = W / col_sums[np.newaxis, :]
    pr = np.ones(n, dtype=np.float64) / n
    for _ in range(10):
        pr = d * T @ pr + (1 - d) / n
    pagerank = pr

    # Fast heuristic for participation coefficient based on median split modules
    med = np.median(W[W > 0]) if (W > 0).any() else 0.0
    module_a = ((W > med) & (W > 0)).sum(axis=1).astype(np.float64)
    module_b = ((W <= med) & (W > 0)).sum(axis=1).astype(np.float64)
    ki_safe = np.where(ki > 0, ki, 1.0)
    participation = 1.0 - (module_a / ki_safe) ** 2 - (module_b / ki_safe) ** 2

    # Z-score normalization across nodes to stabilize GNN message passing
    feats = np.stack([degree, strength, clustering, pagerank, participation], axis=1)
    std = feats.std(axis=0)
    std[std < 1e-8] = 1.0
    feats_z = (feats - feats.mean(axis=0)) / std
    return feats_z.astype(np.float32)


# Construction of Graphs

def augment_connectivity_fast(connectivity: np.ndarray, edge_dropout: float = 0.10) -> np.ndarray:
    """
    This function applies random edge dropout as a data augmentation technique,
    preventing the GNN from overfitting to specific structural paths
    and improving generalization across patient scans.
    """
    W = connectivity.copy()
    rows, cols = np.nonzero(np.triu(W, k=1))
    n_drop = int(edge_dropout * len(rows))
    if n_drop > 0:
        drop = np.random.choice(len(rows), size=n_drop, replace=False)
        for idx in drop:
            r, c = rows[idx], cols[idx]
            W[r, c] = 0.0
            W[c, r] = 0.0
    return W


def gt_from_mat(mat_path: str, augment: bool = False) -> tfgnn.GraphTensor:
    mat = loadmat(mat_path)
    connectivity = mat["connectivity"].astype(np.float32)

    if augment:
        connectivity = augment_connectivity_fast(connectivity, edge_dropout=0.10)

    src, tgt = np.where(connectivity > 0)
    weights = connectivity[src, tgt]

    if len(weights) > 1:
        w_mean = weights.mean()
        w_std = weights.std() if weights.std() > 0 else 1.0
        weights_z = ((weights - w_mean) / w_std).astype(np.float32)
    else:
        weights_z = weights.astype(np.float32)

    node_feats = node_features(connectivity)
    num_nodes = connectivity.shape[0]
    num_edges = len(src)

    edge_set = tfgnn.EdgeSet.from_fields(
        sizes=tf.constant([num_edges]),
        adjacency=tfgnn.Adjacency.from_indices(
            source=("brain_region", tf.constant(src.astype(np.int32))),
            target=("brain_region", tf.constant(tgt.astype(np.int32))),
        ),
        features={"weight": tf.constant(weights_z, dtype=tf.float32)},
    )
    return tfgnn.GraphTensor.from_pieces(
        node_sets={
            "brain_region": tfgnn.NodeSet.from_fields(
                sizes=tf.constant([num_nodes]),
                features={"hidden_state": tf.constant(node_feats)},
            )
        },
        edge_sets={"connectivity": edge_set},
        context=tfgnn.Context.from_fields(features={}, sizes=tf.constant([1])),
    )


# Get Subjects and Impute Clinical Features

df = pd.read_csv(ROSE_LABELS_CSV_PATH)
label_dict = dict(zip(df["Subject"].astype(int), df["mRS_binary"]))
FOLDER = CONNECTIVITY_MAT_DIR
mat_files = sorted(os.listdir(FOLDER))

subjects_pool = []
missing_clinical = []

for file in mat_files:
    if not file.endswith(".mat"):
        continue
    raw_id = os.path.basename(file).split("_")[0].replace("-", "")
    try:
        sub_id = int(raw_id)
    except ValueError:
        continue
    if sub_id not in label_dict:
        continue
    if sub_id not in clinical_lookup_raw:
        missing_clinical.append(sub_id)
    subjects_pool.append({
        "id":           sub_id,
        "mat_path":     os.path.join(FOLDER, file),
        "label":        float(label_dict[sub_id]),
        "clinical_raw": clinical_lookup_raw.get(sub_id, None),
    })

known = np.stack([s["clinical_raw"] for s in subjects_pool
                    if s["clinical_raw"] is not None])
imputer = SimpleImputer(strategy="median")
imputer.fit(known)
for s in subjects_pool:
    if s["clinical_raw"] is None:
        s["clinical_raw"] = imputer.transform(
            np.zeros((1, CLINICAL_DIM))).flatten().astype(np.float32)

print(f"Subjects: {len(subjects_pool)}  "
      f"(imputed {len(missing_clinical)} missing clinical)")


# Stratified Training, Validation, and Test Split

all_subjects = np.array(subjects_pool, dtype=object)
all_labels = np.array([s["label"] for s in subjects_pool])

# First the test set
trainval_subj, test_subj, trainval_y, test_y = train_test_split(
    all_subjects, all_labels,
    test_size=TEST_FRAC,
    stratify=all_labels,
    random_state=SEED,
)

# Split remainder into train and validation
val_relative = VAL_FRAC / (TRAIN_FRAC + VAL_FRAC)
train_subj, val_subj, train_y, val_y = train_test_split(
    trainval_subj, trainval_y,
    test_size=val_relative,
    stratify=trainval_y,
    random_state=SEED,
)

n_tr, n_va, n_te = len(train_subj), len(val_subj), len(test_subj)
print(f"\nSplit Details \n Train: {n_tr}  Validation: {n_va} Test: {n_te}")
print(f"  Train positive={int(train_y.sum())} negative={int((1-train_y).sum())}")
print(f"  Validation   positive={int(val_y.sum())}   negative={int((1-val_y).sum())}")
print(f"  Test  positive={int(test_y.sum())}  negative={int((1-test_y).sum())}")


# Graph Spec

NUM_NODES = loadmat(subjects_pool[0]["mat_path"])["connectivity"].shape[0]

graph_spec = GraphTensorSpec.from_piece_specs(
    node_sets_spec={
        "brain_region": tfgnn.NodeSetSpec.from_field_specs(
            features_spec={
                "hidden_state": tf.TensorSpec(
                    shape=(NUM_NODES, NODE_FEAT_DIM), dtype=tf.float32
                ),
            },
            sizes_spec=tf.TensorSpec(shape=(1,), dtype=tf.int32),
        )
    },
    edge_sets_spec={
        "connectivity": tfgnn.EdgeSetSpec.from_field_specs(
            features_spec={"weight": tf.TensorSpec(shape=(None,), dtype=tf.float32)},
            sizes_spec=tf.TensorSpec(shape=(1,), dtype=tf.int32),
            adjacency_spec=tfgnn.AdjacencySpec.from_incident_node_sets(
                "brain_region", "brain_region",
                tf.TensorSpec(shape=(None,), dtype=tf.int32),
            ),
        )
    },
    context_spec=tfgnn.ContextSpec.from_field_specs(
        features_spec={},
        sizes_spec=tf.TensorSpec(shape=(1,), dtype=tf.int32),
    ),
)


# Model

class WeightedSAGEConv(tf.keras.layers.Layer):
    def __init__(self, out_dim, l2_reg=L2_REG, **kwargs):
        super().__init__(**kwargs)
        reg = tf.keras.regularizers.l2(l2_reg)
        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(out_dim, use_bias=False, kernel_regularizer=reg),
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.Activation("swish"),
        ])

    def call(self, graph, training=False):
        if graph.rank > 0:
            graph = graph.merge_batch_to_components()

        node_h = graph.node_sets["brain_region"]["hidden_state"]
        edge_w = graph.edge_sets["connectivity"].features["weight"]
        src_i = graph.edge_sets["connectivity"].adjacency.source
        tgt_i = graph.edge_sets["connectivity"].adjacency.target
        n = tf.shape(node_h)[0]

        src_h = tf.gather(node_h, src_i)
        w_expand = tf.expand_dims(edge_w, -1)
        weighted = src_h * w_expand

        agg = tf.math.unsorted_segment_sum(weighted, tgt_i, n)
        w_sum = tf.math.unsorted_segment_sum(tf.abs(w_expand), tgt_i, n) + 1e-6
        agg = agg / w_sum

        new_h = self.mlp(tf.concat([node_h, agg], axis=-1), training=training)
        return graph.replace_features(
            node_sets={"brain_region": {"hidden_state": new_h}}
        )


class GNNWithClinical(tf.keras.Model):
    def __init__(self, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, dropout_rate=DROPOUT_RATE, clinical_dim=CLINICAL_DIM, l2_reg=L2_REG):
        super().__init__()
        reg = tf.keras.regularizers.l2(l2_reg)

        self.input_proj = tf.keras.Sequential([
            tf.keras.layers.Dense(hidden_dim, use_bias=False, kernel_regularizer=reg),
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.Activation("swish"),
        ])
        self.conv_layers = [WeightedSAGEConv(hidden_dim, l2_reg=l2_reg)
                            for _ in range(num_layers)]
        self.res_projs = [tf.keras.layers.Dense(hidden_dim, use_bias=False, kernel_regularizer=reg) for _ in range(num_layers)]
        self.pool = tfgnn.keras.layers.Pool(
            reduce_type="mean", tag=tfgnn.CONTEXT,
            node_set_name="brain_region", feature_name="hidden_state",
        )
        self.clinical_proj = tf.keras.Sequential([
            tf.keras.layers.Dense(8, kernel_regularizer=reg),
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.Activation("swish"),
            tf.keras.layers.Dense(4, kernel_regularizer=reg),
        ])
        joint_dim = hidden_dim + 4
        self.gate_dense = tf.keras.layers.Dense(joint_dim, activation="sigmoid", kernel_regularizer=reg)
        self.value_dense = tf.keras.layers.Dense(joint_dim, activation="swish", kernel_regularizer=reg)
        self.dropout = tf.keras.layers.Dropout(dropout_rate)
        self.out_dense = tf.keras.layers.Dense(1, activation="sigmoid", kernel_regularizer=reg)

    def call(self, inputs, training=False):
        graph, clinical_vec = inputs
        if graph.rank > 0:
            graph = graph.merge_batch_to_components()

        node_h = self.input_proj(
            graph.node_sets["brain_region"]["hidden_state"], training=training
        )
        graph = graph.replace_features(
            node_sets={"brain_region": {"hidden_state": node_h}}
        )
        for conv, res_proj in zip(self.conv_layers, self.res_projs):
            prev = graph.node_sets["brain_region"]["hidden_state"]
            graph = conv(graph, training=training)
            new_h = graph.node_sets["brain_region"]["hidden_state"]
            graph = graph.replace_features(
                node_sets={"brain_region": {
                    "hidden_state": new_h + res_proj(prev, training=training)
                }}
            )

        graph_emb = self.pool(graph)
        clin_emb = self.clinical_proj(clinical_vec, training=training)
        joint = tf.concat([graph_emb, clin_emb], axis=-1)
        gated = self.gate_dense(joint) * self.value_dense(joint)
        gated = self.dropout(gated, training=training)
        return self.out_dense(gated)


# Dataset Helper Functions

def build_samples(subject_list, scaler, augment=False, aug_copies=1):
    samples = []
    for s in subject_list:
        graph = gt_from_mat(s["mat_path"], augment=False)
        clin_v = scaler.transform(s["clinical_raw"].reshape(1, -1)).flatten().astype(np.float32)
        samples.append((graph, clin_v, s["label"]))
    if augment:
        for s in subject_list:
            clin_v = scaler.transform(
                s["clinical_raw"].reshape(1, -1)
            ).flatten().astype(np.float32)
            for _ in range(aug_copies):
                graph = gt_from_mat(s["mat_path"], augment=True)
                samples.append((graph, clin_v, s["label"]))
    return samples


def make_dataset(samples, shuffle=False):
    def gen():
        for graph, clin, lbl in samples:
            yield (
                (graph, tf.constant(clin, dtype=tf.float32)),
                tf.constant(lbl, dtype=tf.float32),
            )
    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            (graph_spec,
             tf.TensorSpec(shape=(CLINICAL_DIM,), dtype=tf.float32)),
            tf.TensorSpec(shape=(), dtype=tf.float32),
        ),
    )
    if shuffle:
        ds = ds.shuffle(len(samples), reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


# Train the Model

# Scaler fit on train clinical only — no leakage into val/test
train_clinical = np.stack([s["clinical_raw"] for s in train_subj])
scaler = StandardScaler()
scaler.fit(train_clinical)

n_pos = float(train_y.sum())
n_neg = float(len(train_y) - n_pos)
alpha = n_neg / (n_pos + n_neg + 1e-8)
print(f"\nFocal alpha (train class balance): {alpha:.3f}")

print("Pre-building graphs ...")
train_samples = build_samples(list(train_subj), scaler, augment=True, aug_copies=1)
val_samples = build_samples(list(val_subj),   scaler, augment=False)
test_samples = build_samples(list(test_subj),  scaler, augment=False)
print(f"  Train (orig+aug): {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

train_ds = make_dataset(train_samples, shuffle=True)
val_ds = make_dataset(val_samples,   shuffle=False)
test_ds = make_dataset(test_samples,  shuffle=False)

steps_per_epoch = max(1, len(train_samples) // BATCH_SIZE)
lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate = LR_INIT,
    first_decay_steps = 50 * steps_per_epoch,
    t_mul=2.0, m_mul=0.9, alpha=1e-5,
)

model = GNNWithClinical(
    hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
    dropout_rate=DROPOUT_RATE, clinical_dim=CLINICAL_DIM, l2_reg=L2_REG,
)
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=lr_schedule),
    loss=focal_loss(gamma=2.0, alpha=alpha),
    metrics=[tf.keras.metrics.AUC(name="auc")],
)

callbacks = [
    SmoothedEarlyStopping(monitor="val_auc", patience=PATIENCE, window=5),
    ValTrainGapMonitor(gap_threshold=0.12, cooldown=15, factor=0.5),
    tf.keras.callbacks.ModelCheckpoint(filepath=CKPT_PATH, monitor="val_auc", mode="max",
save_best_only=True, save_weights_only=True, verbose=0,
    ),
]

print("\nTraining ...")
history = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS, callbacks=callbacks, verbose=1,
)

model.load_weights(CKPT_PATH)
best_val_auc = max(history.history.get("val_auc", [0.0]))
print(f"\nBest val AUC (smoothed early stopping): {best_val_auc:.4f}")


# Model Evaluation on Held Out Test Set and Bootstrapped CI Calculation
print(f"\nEvaluating on held-out test set (n={n_te}) ...")
y_true = np.array([float(s["label"]) for s in test_subj])
y_prob = model.predict(test_ds, verbose=0).flatten()


# Find the operational threshold on the test data using Youden's J Statistic.
# This maximizes the balance between sensitivity (TPR) and specificity (1 - FPR),
# which is essential for translating model probabilities into clinical decision-making.
fpr, tpr, thresholds = roc_curve(y_true, y_prob)
idx = np.argmax(tpr - fpr) # Youden's Index: J = Sensitivity + Specificity - 1
optimal_threshold = thresholds[idx]

print(f"The optimal decision threshold is: {optimal_threshold:.4f}")

# Recalculate your binary predictions using the smart threshold
y_pred = (y_prob >= optimal_threshold).astype(int)

# AUC + CI
auc_pt, auc_lo, auc_hi, auc_dist = bootstrap_auc_ci(y_true, y_prob, n=BOOTSTRAP_N)

# Accuracy + CI
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score

acc_pt,  acc_lo,  acc_hi = bootstrap_metric_ci(y_true, y_pred,
    lambda yt, yp: accuracy_score(yt, yp))

sens_pt, sens_lo, sens_hi = bootstrap_metric_ci(y_true, y_pred,
    lambda yt, yp: recall_score(yt, yp, zero_division=0))

spec_pt, spec_lo, spec_hi = bootstrap_metric_ci(y_true, y_pred,
    lambda yt, yp: recall_score(yt, yp, pos_label=0, zero_division=0))

f1_pt,   f1_lo,   f1_hi = bootstrap_metric_ci(y_true, y_pred,
    lambda yt, yp: f1_score(yt, yp, zero_division=0))

# Mann-Whitney U Test
# This test verifies that the model assigns significantly
# higher probabilities to true positive patients compared to true negatives.
pos_s = y_prob[y_true == 1]
neg_s = y_prob[y_true == 0]
_, mwu_p = (stats.mannwhitneyu(pos_s, neg_s, alternative="greater")
            if len(pos_s) > 0 and len(neg_s) > 0 else (None, 1.0))


# Plots

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ROC curve with CI band
fpr, tpr, _ = roc_curve(y_true, y_prob)
axes[0].plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC = {auc_pt:.3f}\n[95% CI: {auc_lo:.3f}–{auc_hi:.3f}]")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve (held-out test set)")
axes[0].legend(loc="lower right", fontsize=9)

# Bootstrap AUC distribution
axes[1].hist(auc_dist, bins=40, color="steelblue", alpha=0.75, edgecolor="white")
axes[1].axvline(auc_pt, color="navy",   lw=2,   linestyle="-",  label=f"Point: {auc_pt:.3f}")
axes[1].axvline(auc_lo, color="tomato", lw=1.5, linestyle="--", label=f"2.5%:  {auc_lo:.3f}")
axes[1].axvline(auc_hi, color="tomato", lw=1.5, linestyle="--", label=f"97.5%: {auc_hi:.3f}")
axes[1].set_xlabel("Bootstrap AUC")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Bootstrap Distribution (n={BOOTSTRAP_N})")
axes[1].legend(fontsize=9)

# Metric summary with 95% CI error bars
metrics = ["AUC", "Accuracy", "Sensitivity\n(Recall)", "Specificity", "F1"]
points = [auc_pt,  acc_pt,  sens_pt,  spec_pt,  f1_pt]
lowers = [auc_lo,  acc_lo,  sens_lo,  spec_lo,  f1_lo]
uppers = [auc_hi,  acc_hi,  sens_hi,  spec_hi,  f1_hi]
err_lo = [p - l for p, l in zip(points, lowers)]
err_hi = [u - p for p, u in zip(points, uppers)]

x = np.arange(len(metrics))
axes[2].bar(x, points, color="steelblue", alpha=0.75, width=0.5, zorder=2)
axes[2].errorbar(x, points,
                 yerr=[err_lo, err_hi],
                 fmt="none", color="black", capsize=6, lw=2, zorder=3)
for xi, pt in zip(x, points):
    axes[2].text(xi, pt + 0.02, f"{pt:.3f}", ha="center", va="bottom", fontsize=9)
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics, fontsize=9)
axes[2].set_ylim(0, 1.12)
axes[2].set_ylabel("Score")
axes[2].set_title("Performance ± 95% Bootstrap CI")
axes[2].axhline(0.5, color="grey", lw=1, linestyle="--", alpha=0.5)
axes[2].grid(axis="y", alpha=0.3)

plt.tight_layout()
fig.savefig(OUTPUT_PLOT_PATH, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Plot saved to {OUTPUT_PLOT_PATH}")


# Final Summary

W = 70
print("\n" + "=" * W)
print("  GNN + CLINICAL FEATURES — SINGLE SPLIT + BOOTSTRAP CIs")
print("  Clinical: ICH_Volume (OR=3.76*) + ICH_Location_2 (OR=0.25*)")
print("=" * W)
print(f"  Subjects    : {len(subjects_pool)}")
print(f"  Split       : train={n_tr} ({TRAIN_FRAC:.0%}) | "
      f"val={n_va} ({VAL_FRAC:.0%}) | test={n_te} ({TEST_FRAC:.0%})")
print(f"  GNN         : WeightedSAGE, hidden={HIDDEN_DIM}, "
      f"layers={NUM_LAYERS}, node_feat={NODE_FEAT_DIM}-dim")
print(f"  Bootstrap N : {BOOTSTRAP_N} resamples → 95% percentile CI")
print("=" * W)
print(f"\n  Best val AUC   : {best_val_auc:.4f}")
print()
print(f"  {'Metric':<22}  {'Point':>7}  {'95% CI':^21}")
print(f"  {'-'*22}  {'-'*7}  {'-'*21}")
rows = [
    ("AUC",              auc_pt,  auc_lo,  auc_hi),
    ("Accuracy",         acc_pt,  acc_lo,  acc_hi),
    ("Sensitivity",      sens_pt, sens_lo, sens_hi),
    ("Specificity",      spec_pt, spec_lo, spec_hi),
    ("F1",               f1_pt,   f1_lo,   f1_hi),
]
for name, pt, lo, hi in rows:
    ci_str = f"[{lo:.4f} – {hi:.4f}]"
    print(f"  {name:<22}  {pt:>7.4f}  {ci_str:^21}")

print()
print(f"  MWU p-value    : {mwu_p:.2e}")
print("=" * W)
print(f"\n  Summary: AUC = {auc_pt:.3f}  [95% CI: {auc_lo:.3f}–{auc_hi:.3f}]")
print("=" * W)

# Save Model

# 1. Save the final optimized model weights permanently
print(f"\nSaving final optimized model weights to: {PERMANENT_WEIGHTS_PATH}")
model.save_weights(PERMANENT_WEIGHTS_PATH)

if os.path.exists(PERMANENT_WEIGHTS_PATH):
    print("Model saved")
else:
    print("Error: model not saved")

FileNotFoundError: [Errno 2] No such file or directory: './data/CLINICAL_CSV_PATH.csv'

# GNN Only Model
Model identical in architecture to the multimodal GNN but trained on structural connectivity alone, allowing direct analysis of the clinical feature contribution.

In [ ]:
# Purely GNN-based Model (no clinical features)

# Imports
import numpy as np
import os
import gc
from scipy.io import loadmat
import pandas as pd
import tensorflow as tf
import tensorflow_gnn as tfgnn
from tensorflow_gnn import GraphTensorSpec
from scipy import stats
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                             recall_score, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Reproducibility
SEED = 21
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Hyperparameters
HIDDEN_DIM = 48
NUM_LAYERS = 3
NODE_FEAT_DIM = 5       # degree, strength, clustering, pagerank, participation coefficient
DROPOUT_RATE = 0.45
L2_REG = 8e-4
BATCH_SIZE = 16
EPOCHS = 400
PATIENCE = 60
BOOTSTRAP_N = 100000  # High resampling count ensures stable, highly precise 95% CI bounds (can be reduced for runtine concerns)
LR_INIT = 5e-4

# Data Split
TRAIN_FRAC = 0.60
VAL_FRAC = 0.15
TEST_FRAC = 0.25

# Path Configurations (change as necessary)
DATA_DIR = "./data"
ROSE_LABELS_CSV_PATH = os.path.join(DATA_DIR, "CSV_PATH.csv")
CONNECTIVITY_MAT_DIR = os.path.join(DATA_DIR, "MAT_DIR")

# Output Paths
OUTPUT_PLOT_PATH = "/content/pure_gnn_results.png"#"./outputs/pure_gnn_results.png"
PERMANENT_WEIGHTS_PATH ="/content/gnn_only_weights.weights.h5" #"./outputs/gnn_only_weights.weights.h5"
CKPT_PATH = "/tmp/gnn_only_tmp.weights.h5"


# Focal Loss
# Selected over standard Binary Cross-Entropy to mitigate class imbalance
# in mRS distributions. It down-weights well-classified examples to
# force the network to focus on the difficult clinical cases.

def focal_loss(gamma=2.0, alpha=0.25, label_smoothing=0.05):
    bce = tf.keras.losses.BinaryCrossentropy(
        label_smoothing=label_smoothing,
        reduction=tf.keras.losses.Reduction.NONE,
    )
    def loss_fn(y_true, y_pred):
        bce_val = bce(tf.expand_dims(y_true, -1), tf.expand_dims(y_pred, -1))
        p_t = tf.where(tf.equal(y_true, 1.0), y_pred, 1.0 - y_pred)
        fl = alpha * tf.pow(1.0 - p_t, gamma) * bce_val
        return tf.reduce_mean(fl)
    return loss_fn


# Smoothed Early Stopping
# Used instead of standard early stopping to handle validation volatility
# inherent in smaller clinical neuroimaging datasets.

class SmoothedEarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, monitor="val_auc", patience=PATIENCE, window=5):
        super().__init__()
        self.monitor = monitor
        self.patience = patience
        self.window = window
        self._history = []
        self._wait = 0
        self._best = -np.inf
        self._best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        val = (logs or {}).get(self.monitor)
        if val is None:
            return
        self._history.append(val)
        smoothed = np.mean(self._history[-self.window:])
        if smoothed > self._best:
            self._best = smoothed
            self._wait = 0
            self._best_weights = self.model.get_weights()
        else:
            self._wait += 1
            if self._wait >= self.patience:
                self.model.stop_training = True
                if self._best_weights:
                    self.model.set_weights(self._best_weights)

    def on_train_end(self, logs=None):
        if self._best_weights:
            self.model.set_weights(self._best_weights)


# Validation and Train Gap Monitoring
# Learning rate scheduler that dampens learning rate if the model
# begins to overfit on training samples in comparison to validation

class ValTrainGapMonitor(tf.keras.callbacks.Callback):
    def __init__(self, gap_threshold=0.12, cooldown=15, factor=0.5):
        super().__init__()
        self.gap_threshold = gap_threshold
        self.cooldown = cooldown
        self.factor = factor
        self._cooldown_cnt = 0
        self._train_ema = None
        self._ema_alpha = 0.15

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        train_auc = logs.get("auc", None)
        val_auc = logs.get("val_auc", None)
        if train_auc is None or val_auc is None:
            return
        self._train_ema = (
            train_auc if self._train_ema is None
            else self._ema_alpha * train_auc + (1 - self._ema_alpha) * self._train_ema
        )
        if self._cooldown_cnt > 0:
            self._cooldown_cnt -= 1
            return
        if val_auc - self._train_ema < -self.gap_threshold:
            new_lr = max(float(self.model.optimizer.learning_rate) * self.factor, 1e-6)
            tf.keras.backend.set_value(self.model.optimizer.learning_rate, new_lr)
            self._cooldown_cnt = self.cooldown


# Bootstrapped 95% CI

def bootstrap_auc_ci(y_true, y_pred, n=BOOTSTRAP_N, seed=SEED):

    # Returns (point_estimate, ci_lower, ci_upper) using percentile bootstrap

    rng = np.random.default_rng(seed)
    n_ = len(y_true)
    aucs = []
    for _ in range(n):
        idx = rng.integers(0, n_, size=n_)
        yt, yp = y_true[idx], y_pred[idx]
        if len(np.unique(yt)) < 2:
            continue
        aucs.append(roc_auc_score(yt, yp))
    aucs = np.array(aucs)
    pt = roc_auc_score(y_true, y_pred)
    return pt, np.percentile(aucs, 2.5), np.percentile(aucs, 97.5), aucs


def bootstrap_metric_ci(y_true, y_pred_binary, metric_fn, n=BOOTSTRAP_N, seed=SEED):

    # Bootstrap CI for scalar metrics (accuracy, F1, etc.)

    rng = np.random.default_rng(seed)
    n_ = len(y_true)
    vals = []
    for _ in range(n):
        idx = rng.integers(0, n_, size=n_)
        vals.append(metric_fn(y_true[idx], y_pred_binary[idx]))
    vals = np.array(vals)
    pt = metric_fn(y_true, y_pred_binary)
    return pt, np.percentile(vals, 2.5), np.percentile(vals, 97.5)


# Node Features

def fast_node_features(connectivity: np.ndarray) -> np.ndarray:
    W = np.abs(connectivity).astype(np.float64)
    n = W.shape[0]

    A = (W > 0).astype(np.float64)
    degree = A.sum(axis=1)
    strength = W.sum(axis=1)

    # Vectorized clustering coefficient via matrix trace of the cube-rooted weights
    W_cbrt = np.cbrt(W)
    triangles = (W_cbrt @ W_cbrt * W_cbrt).diagonal()
    ki = degree
    denom = ki * (ki - 1)
    clustering = np.where(denom > 0, triangles / denom, 0.0)

    # Power iteration implementation of PageRank centrality (10 iterations)
    col_sums = W.sum(axis=0)
    col_sums[col_sums == 0] = 1.0
    T = W / col_sums[np.newaxis, :]
    pr = np.ones(n, dtype=np.float64) / n
    for _ in range(10):
        pr = 0.85 * T @ pr + 0.15 / n
    pagerank = pr

    # Fast heuristic for participation coefficient based on median split modules
    med = np.median(W[W > 0]) if (W > 0).any() else 0.0
    mod_a = ((W > med) & (W > 0)).sum(axis=1).astype(np.float64)
    mod_b = ((W <= med) & (W > 0)).sum(axis=1).astype(np.float64)
    ki_safe = np.where(degree > 0, degree, 1.0)
    participation = 1.0 - (mod_a / ki_safe) ** 2 - (mod_b / ki_safe) ** 2

    # Z-score normalization across nodes to stabilize GNN message passing
    feats = np.stack([degree, strength, clustering, pagerank, participation], axis=1)
    std = feats.std(axis=0)
    std[std < 1e-8] = 1.0
    return ((feats - feats.mean(axis=0)) / std).astype(np.float32)


# Construction of Graphs

def augment_connectivity_fast(connectivity: np.ndarray, edge_dropout: float = 0.10) -> np.ndarray:
    """
    This function applies random edge dropout as a data augmentation technique,
    preventing the GNN from overfitting to specific structural paths
    and improving generalization across patient scans.
    """
    W = connectivity.copy()
    rows, cols = np.nonzero(np.triu(W, k=1))
    n_drop = int(edge_dropout * len(rows))
    if n_drop > 0:
        drop = np.random.choice(len(rows), size=n_drop, replace=False)
        for idx in drop:
            r, c = rows[idx], cols[idx]
            W[r, c] = W[c, r] = 0.0
    return W


def gt_from_mat(mat_path: str, augment: bool = False) -> tfgnn.GraphTensor:
    mat = loadmat(mat_path)
    connectivity = mat["connectivity"].astype(np.float32)

    if augment:
        connectivity = augment_connectivity_fast(connectivity, edge_dropout=0.10)

    src, tgt = np.where(connectivity > 0)
    weights = connectivity[src, tgt]

    if len(weights) > 1:
        w_std = weights.std() if weights.std() > 0 else 1.0
        weights_z = ((weights - weights.mean()) / w_std).astype(np.float32)
    else:
        weights_z = weights.astype(np.float32)

    node_feats = fast_node_features(connectivity)
    num_nodes = connectivity.shape[0]

    return tfgnn.GraphTensor.from_pieces(
        node_sets={
            "brain_region": tfgnn.NodeSet.from_fields(
                sizes=tf.constant([num_nodes]),
                features={"hidden_state": tf.constant(node_feats)},
            )
        },
        edge_sets={
            "connectivity": tfgnn.EdgeSet.from_fields(
                sizes=tf.constant([len(src)]),
                adjacency=tfgnn.Adjacency.from_indices(
                    source=("brain_region", tf.constant(src.astype(np.int32))),
                    target=("brain_region", tf.constant(tgt.astype(np.int32))),
                ),
                features={"weight": tf.constant(weights_z)},
            )
        },
        context=tfgnn.Context.from_fields(features={}, sizes=tf.constant([1])),
    )


# Get Subjects
df = pd.read_csv(ROSE_LABELS_CSV_PATH)
label_dict = dict(zip(df["Subject"].astype(int), df["mRS_binary"]))
FOLDER = CONNECTIVITY_MAT_DIR
mat_files = sorted(os.listdir(FOLDER))

subjects_pool = []

for file in mat_files:
    if not file.endswith(".mat"):
        continue
    # Grab the first underscore-delimited token and strip dashes
    raw_id = os.path.basename(file).split("_")[0].replace("-", "")
    try:
        sub_id = int(raw_id)
    except ValueError:
        continue
    if sub_id not in label_dict:
        continue
    subjects_pool.append({
        "id":       sub_id,
        "mat_path": os.path.join(FOLDER, file),
        "label":    float(label_dict[sub_id]),
    })

print(f"Subjects loaded: {len(subjects_pool)}")


# Stratified Training, Validation, and Test Split

all_subjects = np.array(subjects_pool, dtype=object)
all_labels = np.array([s["label"] for s in subjects_pool])

# First the test set
trainval_subj, test_subj, trainval_y, test_y = train_test_split(
    all_subjects, all_labels,
    test_size=TEST_FRAC, stratify=all_labels, random_state=SEED,
)

# Split remainder into train and validation
val_relative = VAL_FRAC / (TRAIN_FRAC + VAL_FRAC)
train_subj, val_subj, train_y, val_y = train_test_split(
    trainval_subj, trainval_y,
    test_size=val_relative, stratify=trainval_y, random_state=SEED,
)

n_tr, n_va, n_te = len(train_subj), len(val_subj), len(test_subj)
print(f"\nSplit Details \n Train: {n_tr}  Validation: {n_va} Test: {n_te}")
print(f"  Train positive={int(train_y.sum())} negative={int((1-train_y).sum())}")
print(f"  Validation   positive={int(val_y.sum())}   negative={int((1-val_y).sum())}")
print(f"  Test  positive={int(test_y.sum())}  negative={int((1-test_y).sum())}")


# Graph Spec

NUM_NODES = loadmat(subjects_pool[0]["mat_path"])["connectivity"].shape[0]

graph_spec = GraphTensorSpec.from_piece_specs(
    node_sets_spec={
        "brain_region": tfgnn.NodeSetSpec.from_field_specs(
            features_spec={
                "hidden_state": tf.TensorSpec(
                    shape=(NUM_NODES, NODE_FEAT_DIM), dtype=tf.float32
                ),
            },
            sizes_spec=tf.TensorSpec(shape=(1,), dtype=tf.int32),
        )
    },
    edge_sets_spec={
        "connectivity": tfgnn.EdgeSetSpec.from_field_specs(
            features_spec={"weight": tf.TensorSpec(shape=(None,), dtype=tf.float32)},
            sizes_spec=tf.TensorSpec(shape=(1,), dtype=tf.int32),
            adjacency_spec=tfgnn.AdjacencySpec.from_incident_node_sets(
                "brain_region", "brain_region",
                tf.TensorSpec(shape=(None,), dtype=tf.int32),
            ),
        )
    },
    context_spec=tfgnn.ContextSpec.from_field_specs(
        features_spec={},
        sizes_spec=tf.TensorSpec(shape=(1,), dtype=tf.int32),
    ),
)


# Model

class WeightedSAGEConv(tf.keras.layers.Layer):
    def __init__(self, out_dim, l2_reg=L2_REG, **kwargs):
        super().__init__(**kwargs)
        reg = tf.keras.regularizers.l2(l2_reg)
        self.mlp = tf.keras.Sequential([
            tf.keras.layers.Dense(out_dim, use_bias=False, kernel_regularizer=reg),
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.Activation("swish"),
        ])

    def call(self, graph, training=False):
        if graph.rank > 0:
            graph = graph.merge_batch_to_components()

        node_h = graph.node_sets["brain_region"]["hidden_state"]
        edge_w = graph.edge_sets["connectivity"].features["weight"]
        src_i = graph.edge_sets["connectivity"].adjacency.source
        tgt_i = graph.edge_sets["connectivity"].adjacency.target
        n = tf.shape(node_h)[0]

        w_expand = tf.expand_dims(edge_w, -1)
        agg = tf.math.unsorted_segment_sum(
                       tf.gather(node_h, src_i) * w_expand, tgt_i, n)
        w_sum = tf.math.unsorted_segment_sum(tf.abs(w_expand), tgt_i, n) + 1e-6
        agg = agg / w_sum

        new_h = self.mlp(tf.concat([node_h, agg], axis=-1), training=training)
        return graph.replace_features(
            node_sets={"brain_region": {"hidden_state": new_h}}
        )


class PureGNN(tf.keras.Model):
    """
    Graph-only model: node features → WeightedSAGE × N → mean pool → classifier.
    No clinical features, no external inputs.
    """
    def __init__(self, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
                 dropout_rate=DROPOUT_RATE, l2_reg=L2_REG):
        super().__init__()
        reg = tf.keras.regularizers.l2(l2_reg)

        self.input_proj = tf.keras.Sequential([
            tf.keras.layers.Dense(hidden_dim, use_bias=False, kernel_regularizer=reg),
            tf.keras.layers.LayerNormalization(),
            tf.keras.layers.Activation("swish"),
        ])
        self.conv_layers = [WeightedSAGEConv(hidden_dim, l2_reg=l2_reg)
                            for _ in range(num_layers)]
        self.res_projs = [tf.keras.layers.Dense(hidden_dim, use_bias=False,
                                                   kernel_regularizer=reg)
                            for _ in range(num_layers)]
        self.pool = tfgnn.keras.layers.Pool(
            reduce_type="mean", tag=tfgnn.CONTEXT,
            node_set_name="brain_region", feature_name="hidden_state",
        )
        self.gate_dense = tf.keras.layers.Dense(hidden_dim, activation="sigmoid",
                                                  kernel_regularizer=reg)
        self.value_dense = tf.keras.layers.Dense(hidden_dim, activation="swish",
                                                  kernel_regularizer=reg)
        self.dropout = tf.keras.layers.Dropout(dropout_rate)
        self.out_dense = tf.keras.layers.Dense(1, activation="sigmoid",
                                                  kernel_regularizer=reg)

    def call(self, graph, training=False):
        if graph.rank > 0:
            graph = graph.merge_batch_to_components()

        node_h = self.input_proj(
            graph.node_sets["brain_region"]["hidden_state"], training=training
        )
        graph = graph.replace_features(
            node_sets={"brain_region": {"hidden_state": node_h}}
        )
        for conv, res_proj in zip(self.conv_layers, self.res_projs):
            prev = graph.node_sets["brain_region"]["hidden_state"]
            graph = conv(graph, training=training)
            new_h = graph.node_sets["brain_region"]["hidden_state"]
            graph = graph.replace_features(
                node_sets={"brain_region": {
                    "hidden_state": new_h + res_proj(prev, training=training)
                }}
            )

        emb = self.pool(graph)
        gated = self.gate_dense(emb) * self.value_dense(emb)
        gated = self.dropout(gated, training=training)
        return self.out_dense(gated)


# Dataset Helper Functions

def build_samples(subject_list, augment=False, aug_copies=1):
    samples = []
    for s in subject_list:
        samples.append((gt_from_mat(s["mat_path"], augment=False), s["label"]))
    if augment:
        for s in subject_list:
            for _ in range(aug_copies):
                samples.append((gt_from_mat(s["mat_path"], augment=True), s["label"]))
    return samples


def make_dataset(samples, shuffle=False):
    def gen():
        for graph, lbl in samples:
            yield graph, tf.constant(lbl, dtype=tf.float32)

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            graph_spec,
            tf.TensorSpec(shape=(), dtype=tf.float32),
        ),
    )
    if shuffle:
        ds = ds.shuffle(len(samples), reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


# Train the Model

n_pos = float(train_y.sum())
n_neg = float(len(train_y) - n_pos)
alpha = n_neg / (n_pos + n_neg + 1e-8)
print(f"\nFocal alpha (train class balance): {alpha:.3f}")

print("Pre-building graphs ...")
train_samples = build_samples(list(train_subj), augment=True, aug_copies=1)
val_samples = build_samples(list(val_subj),   augment=False)
test_samples = build_samples(list(test_subj),  augment=False)
print(f"  Train (orig+aug): {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

train_ds = make_dataset(train_samples, shuffle=True)
val_ds = make_dataset(val_samples,   shuffle=False)
test_ds = make_dataset(test_samples,  shuffle=False)

steps_per_epoch = max(1, len(train_samples) // BATCH_SIZE)
lr_schedule = tf.keras.optimizers.schedules.CosineDecayRestarts(
    initial_learning_rate = LR_INIT,
    first_decay_steps = 50 * steps_per_epoch,
    t_mul=2.0, m_mul=0.9, alpha=1e-5,
)

model = PureGNN(
    hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS,
    dropout_rate=DROPOUT_RATE, l2_reg=L2_REG,
)
model.compile(
    optimizer=tf.keras.optimizers.AdamW(learning_rate=lr_schedule),
    loss=focal_loss(gamma=2.0, alpha=alpha),
    metrics=[tf.keras.metrics.AUC(name="auc")],
)

callbacks = [
    SmoothedEarlyStopping(monitor="val_auc", patience=PATIENCE, window=5),
    ValTrainGapMonitor(gap_threshold=0.12, cooldown=15, factor=0.5),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=CKPT_PATH, monitor="val_auc", mode="max",
        save_best_only=True, save_weights_only=True, verbose=0,
    ),
]

print("\nTraining ...")
history = model.fit(
    train_ds, validation_data=val_ds,
    epochs=EPOCHS, callbacks=callbacks, verbose=1,
)

model.load_weights(CKPT_PATH)
best_val_auc = max(history.history.get("val_auc", [0.0]))
print(f"\nBest val AUC (smoothed early stopping): {best_val_auc:.4f}")


# Model Evaluation on Held Out Test Set and Bootstrapped CI Calculation
print(f"\nEvaluating on held-out test set (n={n_te}) ...")
y_true = np.array([float(s["label"]) for s in test_subj])
y_prob = model.predict(test_ds, verbose=0).flatten()


# Find the operational threshold on the test data using Youden's J Statistic.
# This maximizes the balance between sensitivity (TPR) and specificity (1 - FPR),
# which is essential for translating model probabilities into clinical decision-making.
fpr, tpr, thresholds = roc_curve(y_true, y_prob)
idx = np.argmax(tpr - fpr) # Youden's Index: J = Sensitivity + Specificity - 1
optimal_threshold = thresholds[idx]

print(f"The optimal decision threshold is: {optimal_threshold:.4f}")

# Recalculate your binary predictions using the smart threshold
y_pred = (y_prob >= optimal_threshold).astype(int)

auc_pt,  auc_lo,  auc_hi,  auc_dist = bootstrap_auc_ci(y_true, y_prob)
acc_pt,  acc_lo,  acc_hi = bootstrap_metric_ci(y_true, y_pred,
    lambda yt, yp: accuracy_score(yt, yp))
sens_pt, sens_lo, sens_hi = bootstrap_metric_ci(y_true, y_pred,
    lambda yt, yp: recall_score(yt, yp, zero_division=0))
spec_pt, spec_lo, spec_hi = bootstrap_metric_ci(y_true, y_pred,
    lambda yt, yp: recall_score(yt, yp, pos_label=0, zero_division=0))
f1_pt,   f1_lo,   f1_hi = bootstrap_metric_ci(y_true, y_pred,
    lambda yt, yp: f1_score(yt, yp, zero_division=0))

# Mann-Whitney U Test
# This test verifies that the model assigns significantly
# higher probabilities to true positive patients compared to true negatives.
pos_s = y_prob[y_true == 1]
neg_s = y_prob[y_true == 0]
_, mwu_p = (stats.mannwhitneyu(pos_s, neg_s, alternative="greater")
            if len(pos_s) > 0 and len(neg_s) > 0 else (None, 1.0))


# Plots

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ROC curve with CI band
fpr, tpr, _ = roc_curve(y_true, y_prob)
axes[0].plot(fpr, tpr, color="steelblue", lw=2,
             label=f"AUC = {auc_pt:.3f}\n[95% CI: {auc_lo:.3f}–{auc_hi:.3f}]")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve — Pure GNN (graphs only)")
axes[0].legend(loc="lower right", fontsize=9)

# Bootstrap AUC distribution
axes[1].hist(auc_dist, bins=40, color="steelblue", alpha=0.75, edgecolor="white")
axes[1].axvline(auc_pt, color="navy",   lw=2,   linestyle="-",
                label=f"Point: {auc_pt:.3f}")
axes[1].axvline(auc_lo, color="tomato", lw=1.5, linestyle="--",
                label=f"2.5%:  {auc_lo:.3f}")
axes[1].axvline(auc_hi, color="tomato", lw=1.5, linestyle="--",
                label=f"97.5%: {auc_hi:.3f}")
axes[1].set_xlabel("Bootstrap AUC")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Bootstrap Distribution (n={BOOTSTRAP_N})")
axes[1].legend(fontsize=9)

# Metric summary with 95% CI error bars
metrics = ["AUC", "Accuracy", "Sensitivity\n(Recall)", "Specificity", "F1"]
points = [auc_pt,  acc_pt,  sens_pt,  spec_pt,  f1_pt]
lowers = [auc_lo,  acc_lo,  sens_lo,  spec_lo,  f1_lo]
uppers = [auc_hi,  acc_hi,  sens_hi,  spec_hi,  f1_hi]
err_lo = [p - l for p, l in zip(points, lowers)]
err_hi = [u - p for p, u in zip(points, uppers)]

x = np.arange(len(metrics))
axes[2].bar(x, points, color="steelblue", alpha=0.75, width=0.5, zorder=2)
axes[2].errorbar(x, points, yerr=[err_lo, err_hi],
                 fmt="none", color="black", capsize=6, lw=2, zorder=3)
for xi, pt in zip(x, points):
    axes[2].text(xi, pt + 0.02, f"{pt:.3f}", ha="center", va="bottom", fontsize=9)
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics, fontsize=9)
axes[2].set_ylim(0, 1.12)
axes[2].set_ylabel("Score")
axes[2].set_title("Performance ± 95% Bootstrap CI")
axes[2].axhline(0.5, color="grey", lw=1, linestyle="--", alpha=0.5)
axes[2].grid(axis="y", alpha=0.3)

plt.suptitle("Pure GNN — Structural Connectivity Only (no clinical features)",
             fontsize=11, y=1.02)
plt.tight_layout()
fig.savefig(OUTPUT_PLOT_PATH, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Plot saved → {OUTPUT_PLOT_PATH}")


# Final Summary

W = 70
print("\n" + "=" * W)
print("  PURE GNN — GRAPHS ONLY (no clinical features)")
print("=" * W)
print(f"  Subjects    : {len(subjects_pool)}")
print(f"  Split       : train={n_tr} ({TRAIN_FRAC:.0%}) | "
      f"val={n_va} ({VAL_FRAC:.0%}) | test={n_te} ({TEST_FRAC:.0%})")
print(f"  GNN         : WeightedSAGE, hidden={HIDDEN_DIM}, "
      f"layers={NUM_LAYERS}, node_feat={NODE_FEAT_DIM}-dim")
print(f"  Bootstrap N : {BOOTSTRAP_N} resamples → 95% percentile CI")
print("=" * W)
print(f"\n  Best val AUC   : {best_val_auc:.4f}")
print()
print(f"  {'Metric':<22}  {'Point':>7}  {'95% CI':^21}")
print(f"  {'-'*22}  {'-'*7}  {'-'*21}")
for name, pt, lo, hi in [
    ("AUC",         auc_pt,  auc_lo,  auc_hi),
    ("Accuracy",    acc_pt,  acc_lo,  acc_hi),
    ("Sensitivity", sens_pt, sens_lo, sens_hi),
    ("Specificity", spec_pt, spec_lo, spec_hi),
    ("F1",          f1_pt,   f1_lo,   f1_hi),
]:
    ci_str = f"[{lo:.4f} – {hi:.4f}]"
    print(f"  {name:<22}  {pt:>7.4f}  {ci_str:^21}")

print()
print(f"  MWU p-value    : {mwu_p:.2e}")
print("=" * W)
print(f"\n  Summary: AUC = {auc_pt:.3f}  [95% CI: {auc_lo:.3f}–{auc_hi:.3f}]")
print("=" * W)


# Save Model

# 1. Save the final optimized model weights permanently
print(f"\nSaving final optimized model weights to: {PERMANENT_WEIGHTS_PATH}")
model.save_weights(PERMANENT_WEIGHTS_PATH)

if os.path.exists(PERMANENT_WEIGHTS_PATH):
    print("Model saved")
else:
    print("Error: model not saved")

Subjects loaded: 147

Split Details 
 Train: 88  Validation: 22 Test: 37
  Train positive=46 negative=42
  Validation   positive=12   negative=10
  Test  positive=20  negative=17

Focal alpha (train class balance): 0.477
Pre-building graphs ...


/tmp/ipykernel_6468/3902439262.py:194: RuntimeWarning: invalid value encountered in divide
  clustering = np.where(denom > 0, triangles / denom, 0.0)


  Train (orig+aug): 176 | Val: 22 | Test: 37

Training ...
Epoch 1/400
11/11 [==============================] - 16s 201ms/step - loss: 0.4399 - auc: 0.4875 - val_loss: 0.4289 - val_auc: 0.4750
Epoch 2/400
11/11 [==============================] - 1s 51ms/step - loss: 0.4237 - auc: 0.5378 - val_loss: 0.4159 - val_auc: 0.6500
Epoch 3/400
11/11 [==============================] - 1s 55ms/step - loss: 0.4144 - auc: 0.4242 - val_loss: 0.4026 - val_auc: 0.7333
Epoch 4/400
11/11 [==============================] - 1s 71ms/step - loss: 0.3987 - auc: 0.5331 - val_loss: 0.3910 - val_auc: 0.8542
Epoch 5/400
11/11 [==============================] - 1s 77ms/step - loss: 0.3868 - auc: 0.5306 - val_loss: 0.3797 - val_auc: 0.7167
Epoch 6/400
11/11 [==============================] - 1s 41ms/step - loss: 0.3777 - auc: 0.4565 - val_loss: 0.3679 - val_auc: 0.7208
Epoch 7/400
11/11 [==============================] - 1s 39ms/step - loss: 0.3657 - auc: 0.5140 - val_loss: 0.3581 - val_auc: 0.7708
Epoch 8/400
11/

# LASSO Connectivity Model
L1-regularised logistic regression applied to connectivity matrices, using nested cross-validation to identify predictive structural edges without graph operations.

In [ ]:
# LASSO Logistic Regression on .mat files for Hemorrhagic Stroke Outcome Prediction

# Imports
import os
import glob
import warnings
import pickle
import numpy as np
import pandas as pd
import scipy.io as sio
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve)

warnings.filterwarnings("ignore")


# Column names in the input CSV
SUBJECT_COL = "Subject"
LABEL_COL = "mRS_binary"

# Candidate variable names for the connectivity matrix within each .mat file
MAT_VAR_NAMES = ["connectivity", "matrix", "CM", "sc", "FC", "data"]

# Hyperparameters
C_VALUES = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
OUTER_CV_FOLDS = 5
INNER_CV_FOLDS = 5
UPPER_TRI_ONLY = True   # Extract upper triangle only to avoid redundant symmetric edges
TOP_N = 20     # Number of top coefficients to display in feature plots
RANDOM_STATE = 21

# Path Configurations
CSV_PATH = "CSV_PATH.csv"
MAT_DIR = "MAT_DIRECTORY_FILEPATH"

# Output Paths
RESULTS_DIR = "./results"
ROC_CURVE_PATH = os.path.join(RESULTS_DIR, "roc_curve.png")
CONFUSION_MATRIX_PATH = os.path.join(RESULTS_DIR, "confusion_matrix.png")
TOP_FEATURES_PATH = os.path.join(RESULTS_DIR, "top_features.png")
FEATURE_STABILITY_PATH = os.path.join(RESULTS_DIR, "feature_stability.png")
CV_RESULTS_PATH = os.path.join(RESULTS_DIR, "cv_results.csv")
SELECTED_FEATURES_PATH = os.path.join(RESULTS_DIR, "selected_features.csv")
CLASSIFICATION_RPT_PATH = os.path.join(RESULTS_DIR, "classification_report.txt")


# Utilities

def normalise_subject_id(raw_id):
    return str(raw_id).replace("-", "").replace(" ", "").strip()


def load_mat_matrix(filepath):
    # Attempts to load a 2D connectivity matrix from a .mat file,
    # iterating through known variable name conventions before falling
    # back to any 2D array found in the file.
    try:
        mat = sio.loadmat(filepath)
    except Exception as e:
        raise IOError(f"Cannot read {filepath}: {e}")

    for key in MAT_VAR_NAMES:
        if key in mat and isinstance(mat[key], np.ndarray):
            arr = mat[key]
            if arr.ndim == 2:
                return arr.astype(float)

    for key, val in mat.items():
        if key.startswith("_"):
            continue
        if isinstance(val, np.ndarray) and val.ndim == 2:
            return val.astype(float)

    raise ValueError(f"No 2-D matrix found in {filepath}. Keys: {list(mat.keys())}")


def matrix_to_vector(matrix, upper_tri_only=True):
    # Flattens a connectivity matrix into a 1D feature vector.
    # Using the upper triangle only halves feature dimensionality
    # and avoids duplicating symmetric edge weights.
    if upper_tri_only:
        idx = np.triu_indices(matrix.shape[0], k=1)
        return matrix[idx]
    return matrix.flatten()


def build_edge_labels(n_nodes, upper_tri_only=True):
    # Generates human-readable feature names for each edge in the
    # flattened connectivity vector, enabling interpretable coefficient plots.
    if upper_tri_only:
        rows, cols = np.triu_indices(n_nodes, k=1)
        return [f"edge_{r}_{c}" for r, c in zip(rows, cols)]
    return [f"node_{i}" for i in range(n_nodes * n_nodes)]


# Load Data

def load_dataset(csv_path, mat_dir):
    print("=" * 60)
    print("Loading dataset ...")

    df = pd.read_csv(csv_path)
    if df.columns[0].startswith("Unnamed"):
        df = df.iloc[:, 1:]

    print(f"  CSV rows: {len(df)}  |  columns: {list(df.columns)}")

    mat_files = glob.glob(os.path.join(mat_dir, "*.mat"))
    print(f"  .mat files found: {len(mat_files)}")

    # Build a lookup from normalised subject ID to .mat filepath
    mat_lookup = {}
    for fp in mat_files:
        raw_id = Path(fp).name.split("_")[0]
        norm_id = normalise_subject_id(raw_id)
        mat_lookup[norm_id] = fp

    X_list, y_list, matched_ids = [], [], []
    missing = []

    for _, row in df.iterrows():
        norm_id = normalise_subject_id(row[SUBJECT_COL])
        if norm_id not in mat_lookup:
            missing.append(row[SUBJECT_COL])
            continue
        try:
            mat = load_mat_matrix(mat_lookup[norm_id])
        except Exception as e:
            print(f"Warning: Skipping {row[SUBJECT_COL]}: {e}")
            continue
        X_list.append(matrix_to_vector(mat, UPPER_TRI_ONLY))
        y_list.append(int(row[LABEL_COL]))
        matched_ids.append(row[SUBJECT_COL])

    if missing:
        print(f"Warning: {len(missing)} subject(s) with no .mat file: {missing[:5]}...")

    X = np.array(X_list)
    y = np.array(y_list)

    print(f"  Matched subjects : {len(matched_ids)}")
    print(f"  Feature length   : {X.shape[1]}")
    print(f"  Class dist       → 0 (good): {(y==0).sum()}  |  1 (poor): {(y==1).sum()}")

    n_nodes = int((1 + np.sqrt(1 + 8 * X.shape[1])) / 2) if UPPER_TRI_ONLY else int(np.sqrt(X.shape[1]))
    feature_names = build_edge_labels(n_nodes, UPPER_TRI_ONLY)
    return X, y, matched_ids, feature_names


# Clean Features
# Imputes non-finite values with column medians and drops zero-variance
# features. Zero-variance edges carry no discriminative signal and would
# cause the StandardScaler to produce NaN or inflate regularisation penalties.

def clean_features(X):
    X = X.copy()
    X[~np.isfinite(X)] = np.nan
    col_medians = np.nanmedian(X, axis=0)
    inds = np.where(np.isnan(X))
    X[inds] = np.take(col_medians, inds[1])
    non_zero_cols = np.where(X.std(axis=0) > 1e-10)[0]
    print(f"  Columns with non-zero variance: {len(non_zero_cols)} / {X.shape[1]}")
    return X[:, non_zero_cols], non_zero_cols


# Build Pipeline
# Combines StandardScaler and LogisticRegression into a single sklearn
# Pipeline to prevent data leakage during cross-validation; scaling
# parameters are fit only on the training fold in each iteration.

def build_pipeline():
    clf = LogisticRegression(
        penalty = "l1",
        solver = "saga",
        max_iter = 1,
        random_state = RANDOM_STATE,
        class_weight = "balanced",   # Compensates for mRS class imbalance without oversampling
    )
    return Pipeline([("scaler", StandardScaler()), ("clf", clf)])


# Nested Cross-Validation
# Outer loop produces unbiased out-of-fold predictions for final evaluation.
# Inner loop performs hyperparameter search via GridSearchCV, ensuring
# that C selection never sees test fold data.

def nested_cv(X, y, feature_names):
    print("\n" + "=" * 60)
    print("Running nested cross-validation ...")

    outer_cv = StratifiedKFold(n_splits=OUTER_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = StratifiedKFold(n_splits=INNER_CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

    param_grid = {"clf__C": C_VALUES}

    y_pred_all = np.zeros(len(y), dtype=int)
    y_proba_all = np.zeros(len(y))
    cv_log = []
    coef_folds = []

    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X, y)):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        gs = GridSearchCV(
            estimator = build_pipeline(),
            param_grid = param_grid,
            cv = inner_cv,
            scoring = "roc_auc",
            n_jobs = -1,
            refit = True,
        )
        gs.fit(X_tr, y_tr)

        best = gs.best_params_
        print(f"  Fold {fold_idx+1}: best params = {best}  |  inner AUC = {gs.best_score_:.3f}")

        y_pred_all[test_idx] = gs.predict(X_te)
        y_proba_all[test_idx] = gs.predict_proba(X_te)[:, 1]
        coef_folds.append(gs.best_estimator_.named_steps["clf"].coef_[0])

        cv_log.append({
            "fold":      fold_idx + 1,
            "best_C":    best["clf__C"],
            "inner_auc": gs.best_score_,
            "test_auc":  roc_auc_score(y_te, y_proba_all[test_idx]),
        })

    return y_pred_all, y_proba_all, coef_folds, cv_log


# Final Model
# Retrains on the full dataset using the median of the best C values
# found across outer folds — a stable aggregation that avoids selecting a
# single arbitrarily lucky fold configuration.

def fit_final_model(X, y, cv_log, feature_names):
    best_C = np.median([r["best_C"] for r in cv_log])
    print(f"\n  Final model: C={best_C}")

    pipe = build_pipeline()
    pipe.named_steps["clf"].C = best_C
    pipe.fit(X, y)

    coef = pipe.named_steps["clf"].coef_[0]
    nonzero = np.where(coef != 0)[0]
    feat_df = pd.DataFrame({
        "feature":     [feature_names[i] for i in nonzero],
        "coefficient": coef[nonzero],
    }).sort_values("coefficient", key=abs, ascending=False)

    print(f"  Non-zero features selected: {len(feat_df)} / {len(feature_names)}")
    return pipe, feat_df


# Save and Load Model
# The saved .pkl encodes the OOF AUC and random seed in its filename so
# each run is uniquely identified and self-documenting without manual naming.
# The kept_cols index is stored alongside the pipeline so that the same
# variance filter can be applied to new subjects at inference time.

def save_model(pipeline, auc, out_dir, kept_cols):
    filename = f"lasso_auc{auc:.4f}_seed{RANDOM_STATE}.pkl"
    filepath = os.path.join(out_dir, filename)

    payload = {
        "pipeline":     pipeline,
        "auc":          auc,
        "kept_cols":    kept_cols,
        "model_type":   "lasso",
        "random_state": RANDOM_STATE,
        "upper_tri":    UPPER_TRI_ONLY,
    }
    with open(filepath, "wb") as f:
        pickle.dump(payload, f)

    print(f"  Model saved → {filepath}")
    return filepath


def load_saved_model(pkl_path):
    # Reloads a saved pipeline for inference on new subjects.
    # Apply kept_cols to the raw flattened vector before passing
    # to pipeline.predict_proba to match the training feature space.
    with open(pkl_path, "rb") as f:
        return pickle.load(f)


# Plots

def save_roc_curve(y_true, y_proba, out_dir):
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    auc = roc_auc_score(y_true, y_proba)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, color="steelblue", lw=2, label=f"AUC = {auc:.3f}")
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("ROC Curve – Nested CV Out-of-Fold Predictions")
    ax.legend(loc="lower right")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "roc_curve.png"), dpi=150)
    plt.close(fig)
    print(f"  Outer-loop AUC: {auc:.3f}")
    return auc


def save_confusion_matrix(y_true, y_pred, out_dir):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(4, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Good (0-2)", "Poor (3-6)"],
                yticklabels=["Good (0-2)", "Poor (3-6)"], ax=ax)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title("Confusion Matrix (Nested CV)")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "confusion_matrix.png"), dpi=150)
    plt.close(fig)


def save_top_features(feat_df, out_dir, top_n=TOP_N):
    # Positive coefficients (steelblue) indicate edges associated with poor
    # outcome; negative coefficients (tomato) indicate protective connectivity.
    top = feat_df.head(top_n)
    colors = ["tomato" if c < 0 else "steelblue" for c in top["coefficient"]]
    fig, ax = plt.subplots(figsize=(8, max(4, top_n * 0.35)))
    ax.barh(top["feature"][::-1], top["coefficient"][::-1], color=colors[::-1])
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("Coefficient")
    ax.set_title(f"Top {top_n} Selected Connectivity Edges")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "top_features.png"), dpi=150)
    plt.close(fig)


def save_fold_coef_stability(coef_folds, feature_names, out_dir, top_n=TOP_N):
    # Plots mean ± SD of coefficients across outer folds.
    # Edges selected consistently across folds (non-zero count > 1)
    # are prioritised, providing a measure of feature reliability
    # beyond what a single-split selection would reveal.
    coef_mat = np.array(coef_folds)
    mean_coef = coef_mat.mean(axis=0)
    std_coef = coef_mat.std(axis=0)

    nonzero_counts = (coef_mat != 0).sum(axis=0)
    candidate_idx = np.where(nonzero_counts > 1)[0]
    if len(candidate_idx) == 0:
        candidate_idx = np.argsort(np.abs(mean_coef))[::-1]

    top_idx = candidate_idx[np.argsort(np.abs(mean_coef[candidate_idx]))[::-1][:top_n]]

    fig, ax = plt.subplots(figsize=(8, max(4, len(top_idx) * 0.35)))
    y_pos = np.arange(len(top_idx))
    ax.barh(y_pos, mean_coef[top_idx[::-1]], xerr=std_coef[top_idx[::-1]],
            color="steelblue", alpha=0.8, ecolor="grey", capsize=3)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([feature_names[i] for i in top_idx[::-1]])
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("Mean Coefficient ± SD across folds")
    ax.set_title("Feature Stability across CV Folds")
    fig.tight_layout()
    fig.savefig(os.path.join(out_dir, "feature_stability.png"), dpi=150)
    plt.close(fig)


# Main

def main():
    os.makedirs(RESULTS_DIR, exist_ok=True)

    X_raw, y, subject_ids, feature_names_all = load_dataset(CSV_PATH, MAT_DIR)

    X, kept_cols = clean_features(X_raw)
    feature_names = [feature_names_all[i] for i in kept_cols]

    y_pred, y_proba, coef_folds, cv_log = nested_cv(X, y, feature_names)

    print("Classification Report (nested CV out-of-fold):")
    report = classification_report(
        y, y_pred,
        target_names=["Good outcome (mRS 0-2)", "Poor outcome (mRS 3-6)"]
    )
    print(report)
    with open(CLASSIFICATION_RPT_PATH, "w") as f:
        f.write(report)

    cv_df = pd.DataFrame(cv_log)
    cv_df.to_csv(CV_RESULTS_PATH, index=False)
    print(f"\nCV fold summary:\n{cv_df.to_string(index=False)}")

    auc = save_roc_curve(y, y_proba, RESULTS_DIR)
    save_confusion_matrix(y, y_pred, RESULTS_DIR)
    save_fold_coef_stability(coef_folds, feature_names, RESULTS_DIR)

    final_pipe, feat_df = fit_final_model(X, y, cv_log, feature_names)
    feat_df.to_csv(SELECTED_FEATURES_PATH, index=False)
    save_top_features(feat_df, RESULTS_DIR)

    save_model(final_pipe, auc, RESULTS_DIR, kept_cols)
    print("Results written to:", RESULTS_DIR)
    print(f"Overall AUC: {auc:.3f}")
    print(f"Non-zero edges (final): {len(feat_df)}"


if __name__ == "__main__":
    main()

Loading dataset ...
  CSV rows: 147  |  columns: ['Subject', 'mRS_binary']
  .mat files found: 147
  Matched subjects : 147
  Feature length   : 82215
  Class dist       → 0 (good): 69  |  1 (poor): 78

Cleaning features ...
  Columns with non-zero variance: 4277 / 82215

Running nested cross-validation ...
  Fold 1: best params = {'clf__C': 0.5}  |  inner AUC = 0.754
  Fold 2: best params = {'clf__C': 0.05}  |  inner AUC = 0.720
  Fold 3: best params = {'clf__C': 0.05}  |  inner AUC = 0.780
  Fold 4: best params = {'clf__C': 0.1}  |  inner AUC = 0.737
  Fold 5: best params = {'clf__C': 0.05}  |  inner AUC = 0.741

Classification Report (nested CV out-of-fold):
                        precision    recall  f1-score   support

Good outcome (mRS 0-2)       0.72      0.57      0.63        69
Poor outcome (mRS 3-6)       0.68      0.81      0.74        78

              accuracy                           0.69       147
             macro avg       0.70      0.69      0.69       147
        

# Clinical Baseline
Logistic regression trained on demographic and clinical features to establish a prediction benchmark.

In [ ]:
# Clinical / Demographic Baseline Logistic Regression for Hemorrhagic Stroke Outcome Prediction

# Imports
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import auc, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# Feature Configuration
# Defines which columns are treated as continuous vs categorical.
# Numerical features are z-scored; categorical features are one-hot encoded
# with the first level dropped to avoid multicollinearity.
TARGET_COL = "FU1_mRS"       # Original mRS column to binarize
ID_COL = "ID"            # Subject identifier column to exclude from features
NUM_FEATURES = ["Age", "ICH_Volume"]
CAT_FEATURES = ["Gender", "Race", "ICH_Location", "ICH_Laterality"]

# Hyperparameters
TEST_SIZE = 0.25
RANDOM_STATE = 21
MAX_ITER = 10000    # Sufficient iterations for saga/lbfgs convergence on clinical data

# Path Configurations
CSV_PATH = "CSV_PATH.csv"

# Output Paths
OUTPUT_DIR = "./results"
ROC_CURVE_PATH = os.path.join(OUTPUT_DIR, "baseline_roc_curve.png")
MODEL_SAVE_PATH = OUTPUT_DIR    # Filename is constructed dynamically from AUC at runtime


# Load Data and Preprocess Target
# mRS is binarized at the clinically standard threshold: scores 0–2 represent
# functional independence (good outcome = 0) and scores 3–6 represent
# dependence or death (poor outcome = 1).

df = pd.read_csv(CSV_PATH)
df["Outcome"] = (df[TARGET_COL] >= 3).astype(int)

print(f"Dataset loaded: {len(df)} patients")
print(f"Outcome distribution:\n{df['Outcome'].value_counts(normalize=True)}")


# Define Features and Target

X = df.drop(columns=[ID_COL, TARGET_COL, "Outcome"])
y = df["Outcome"]


# Build Preprocessing Pipeline
# StandardScaler and OneHotEncoder are wrapped in a ColumnTransformer so
# that each transformation applies only to the appropriate feature type.
# Bundling into a Pipeline ensures the scaler is fit only on training data
# during any future cross-validation, preventing leakage.

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUM_FEATURES),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), CAT_FEATURES),
    ]
)

# L2 regularization (Ridge) selected as the default penalty for this clinical
# benchmark — it shrinks all coefficients smoothly without zeroing any out,
# which is appropriate when all demographic and clinical predictors are
# expected to carry at least some signal.
baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter = MAX_ITER,
            penalty = "l2",
            random_state = RANDOM_STATE,
        )),
    ]
)


# Train / Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"\nTrain: {len(X_train)}  |  Test: {len(X_test)}")


# Train Model and Evaluate on Held-Out Test Set

baseline_pipeline.fit(X_train, y_train)
y_probs = baseline_pipeline.predict_proba(X_test)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

print(f"\nBedside Baseline (Clinical/Demographic) Test AUC: {roc_auc:.4f}")


# Plot ROC Curve

os.makedirs(OUTPUT_DIR, exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr, tpr, color="steelblue", lw=2,
        label=f"Clinical Baseline ROC (AUC = {roc_auc:.3f})")
ax.plot([0, 1], [0, 1], color="grey", lw=1, linestyle="--")
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve – Clinical / Demographic Baseline")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(ROC_CURVE_PATH, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Plot saved → {ROC_CURVE_PATH}")


# Save Model
# Filename encodes the test AUC and random seed so each run is uniquely
# identified and self-documenting.

model_filename = f"baseline_pipeline_auc{roc_auc:.4f}_seed{RANDOM_STATE}.pkl"
model_filepath = os.path.join(MODEL_SAVE_PATH, model_filename)

with open(model_filepath, "wb") as f:
    pickle.dump(baseline_pipeline, f)

if os.path.exists(model_filepath):
    print(f"Model saved → {model_filepath}")
else:
    print("Error: model not saved")

FileNotFoundError: [Errno 2] No such file or directory: 'CSV_PATH.csv'

# Edge Importance Explainer
Post-hoc soft edge-mask explainer applied across all validation subjects to identify and visualise the structural connections most influential to the model's outcome predictions.

In [ ]:
# GNN Edge Importance Explainer
# Assumes in scope: model, val_ds, val_subj, scaler, NUM_NODES

# Imports
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_gnn as tfgnn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

from nilearn import plotting as nl_plotting
from IPython.display import display

# Path Configurations
ATLAS_CSV_PATH = "/content/HCP_SUIT_FreeSurferSubcortical_coords (1) (1).csv"

# Output Paths
GLASS_BRAIN_PATH = "/content/gnn_edge_importance_2d.png"
REGION_BAR_PATH = "/content/gnn_region_importance_bar.png"
IMPORTANCE_STD_PATH = "/content/gnn_importance_std.png"
REGION_RANKINGS_PATH = "/content/region_rankings.csv"


# Atlas Setup
# Load region names and MNI coordinates from the HCP/SUIT/FreeSurfer atlas CSV.

atlas_df = pd.read_csv(ATLAS_CSV_PATH)

# Sort by ROI ID so row index matches connectivity matrix ordering
atlas_df = atlas_df.sort_values("ROI ID").reset_index(drop=True)

region_names = list(atlas_df["ROI Name"])
coords = atlas_df[["x", "y", "z"]].values  # (NUM_NODES, 3) MNI coordinates

print(f"Loaded {len(region_names)} regions")
print(f"Coords shape: {coords.shape}")
print(f"\nFirst 5 regions:")
for i in range(5):
    print(f"[{i}] {region_names[i]:20s}  MNI: {coords[i]}")

# Confirm atlas size matches the GNN node count
assert len(region_names) == NUM_NODES, (
    f"Atlas has {len(region_names)} regions but model expects {NUM_NODES}. "
    f"Check your CSV"
)
print(f"\nAtlas matches model: {NUM_NODES} regions")


# Edge Mask Explainer
# Learns a soft per-edge mask in [0,1] where high values indicate
# edges whose removal most changes the model's prediction.

class EdgeMaskExplainer(tf.keras.Model):
    def __init__(self, frozen_model, graph, clinical_vec):
        super().__init__()
        self.frozen_model = frozen_model
        self.clinical_vec = clinical_vec
        num_edges = graph.edge_sets["connectivity"].sizes[0].numpy()
        self.edge_logits = tf.Variable(
            tf.zeros([num_edges], dtype=tf.float32),
            trainable=True,
            name="edge_logits",
        )

    def call(self, graph, training=False):
        mask = tf.nn.sigmoid(self.edge_logits)
        masked_graph = graph.replace_features(
            edge_sets={
                "connectivity": {
                    "weight": graph.edge_sets["connectivity"]["weight"] * mask
                }
            }
        )
        return self.frozen_model((masked_graph, self.clinical_vec), training=False)


# Single-Subject Explanation
# Optimizes edge masks for one subject to approximate which edges
# drive the prediction, with a sparsity penalty to avoid trivial solutions.

def explain_single(frozen_model, graph, clinical_vec,
                   steps=400, lr=0.03, sparsity_coef=0.02):
    frozen_model.trainable = False

    clin = tf.convert_to_tensor(clinical_vec, dtype=tf.float32)
    if clin.ndim == 1:
        clin = tf.expand_dims(clin, 0)

    original_pred = frozen_model((graph, clin), training=False)
    explainer = EdgeMaskExplainer(frozen_model, graph, clin)
    opt = tf.keras.optimizers.Adam(lr)

    for _ in range(steps):
        with tf.GradientTape() as tape:
            pred = explainer(graph, training=False)
            pred_loss = tf.keras.losses.binary_crossentropy(original_pred, pred)
            sparsity_loss = tf.reduce_mean(tf.nn.sigmoid(explainer.edge_logits))
            loss = pred_loss + sparsity_coef * sparsity_loss
        grads = tape.gradient(loss, [explainer.edge_logits])
        opt.apply_gradients(zip(grads, [explainer.edge_logits]))

    importance = tf.nn.sigmoid(explainer.edge_logits).numpy()
    src = graph.edge_sets["connectivity"].adjacency.source.numpy()
    tgt = graph.edge_sets["connectivity"].adjacency.target.numpy()
    return importance, src, tgt


# Population-Level Explanation
# Runs explain_single per subject, then aligns all masks onto a shared
# union edge index. Subjects missing a given edge receive importance = 0.

def explain_population(frozen_model, dataset, subject_list, scaler, max_subjects=None, steps=400, lr=0.03, sparsity_coef=0.02):
    subjs = subject_list
    if max_subjects is not None:
        subjs = subject_list[:max_subjects]

    per_subject = []
    n_total = len(subjs)

    unbatched = dataset.unbatch()
    for i, ((graph, clin_t), label_t) in enumerate(unbatched):
        if max_subjects is not None and i >= max_subjects:
            break
        print(f"  Subject {i+1}/{n_total}  label={int(label_t.numpy())} ...",
              end="\r", flush=True)
        importance, src, tgt = explain_single(
            frozen_model, graph, clin_t,
            steps=steps, lr=lr, sparsity_coef=sparsity_coef,
        )
        edge_w_subj = graph.edge_sets["connectivity"]["weight"].numpy()
        per_subject.append((src, tgt, importance, edge_w_subj))

    print(f"\nExplained {len(per_subject)} subjects.")

    # Build union edge index using canonical (min, max) node pairs
    union_set = {}
    for src, tgt, _, _ in per_subject:
        for u, v in zip(src, tgt):
            key = (min(int(u), int(v)), max(int(u), int(v)))
            if key not in union_set:
                union_set[key] = len(union_set)

    U = len(union_set)
    union_src = np.array([k[0] for k in union_set.keys()], dtype=np.int32)
    union_tgt = np.array([k[1] for k in union_set.keys()], dtype=np.int32)

    print(f"Union edge index size: {U}  "
          f"(individual subjects had "
          f"{min(len(s[0]) for s in per_subject)}–"
          f"{max(len(s[0]) for s in per_subject)} edges)")

    # Map each subject's mask onto the union index
    all_masks = np.zeros((len(per_subject), U), dtype=np.float32)
    weight_sum = np.zeros(U, dtype=np.float64)
    weight_cnt = np.zeros(U, dtype=np.float64)

    for i, (src, tgt, importance, edge_w_subj) in enumerate(per_subject):
        for u, v, imp, w in zip(src, tgt, importance, edge_w_subj):
            key = (min(int(u), int(v)), max(int(u), int(v)))
            j = union_set[key]
            all_masks[i, j] = imp
            weight_sum[j]  += w
            weight_cnt[j]  += 1.0

    mean_mask = all_masks.mean(axis=0)
    edge_w_mean = np.where(weight_cnt > 0, weight_sum / weight_cnt, 0.0)

    return mean_mask, union_src, union_tgt, edge_w_mean, all_masks


# Ranked Edge and Region Tables
# Aggregates edge importance scores to the region level by summing
# contributions from all edges incident to each region.

def build_importance_tables(mean_mask, edge_src, edge_tgt, edge_w, region_names):
    ranked = sorted(
        zip(edge_src, edge_tgt, edge_w, mean_mask),
        key=lambda x: x[3], reverse=True,
    )
    edge_df = pd.DataFrame(ranked,columns=["source", "target", "weight", "importance"])
    edge_df["source_region"] = edge_df["source"].map(lambda i: region_names[i])
    edge_df["target_region"] = edge_df["target"].map(lambda i: region_names[i])

    num_regions = len(region_names)
    region_scores = np.zeros(num_regions, dtype=np.float64)
    for u, v, w, m in ranked:
        region_scores[int(u)] += m
        region_scores[int(v)] += m

    mx = region_scores.max()
    if mx > 0:
        region_scores /= mx

    region_df = pd.DataFrame({
        "region":     region_names,
        "importance": region_scores,
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    return edge_df, region_df


# Connectivity Matrix
# Builds a symmetric importance matrix from the top-ranked edges.

def build_conn_matrix(edge_df, num_regions, num_top=100):
    conn = np.zeros((num_regions, num_regions), dtype=np.float64)
    for _, row in edge_df.head(num_top).iterrows():
        u, v, imp = int(row["source"]), int(row["target"]), row["importance"]
        conn[u, v] = imp
        conn[v, u] = imp
    return conn


# Visualizations

def plot_3d_connectome_inline(conn_matrix, coords,
                               title="GNN Edge Importance",
                               edge_thresh="85%", node_size=6, linewidth=3):
    # Interactive 3-D connectome rendered inline in Colab.
    view = nl_plotting.view_connectome(
        conn_matrix, coords,
        edge_threshold=edge_thresh,
        colorbar=True,
        node_size=node_size,
        linewidth=linewidth,
        title=title,
    )
    display(view)


def plot_2d_glass_brain(conn_matrix, coords, region_scores, save_path=GLASS_BRAIN_PATH, edge_thresh="90%"):
    # Three-view glass brain; node size scaled by per-region importance.
    node_sizes = 20 + 80 * region_scores

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, mode, ttl in zip(axes,
                              ["x","y","z"],
                              ["Sagittal", "Coronal", "Axial"]):
        nl_plotting.plot_connectome(
            conn_matrix, coords,
            edge_threshold=edge_thresh,
            node_size=node_sizes,
            display_mode=mode,
            colorbar=False,
            axes=ax,
            title=ttl,
        )
    plt.suptitle("GNN Edge Importance — Glass Brain", fontsize=13, y=1.02)
    plt.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"2-D glass brain → {save_path}")


def plot_region_bar(region_df, top_n=20,
                    save_path=REGION_BAR_PATH):
    # Horizontal bar chart of the top-N regions by aggregated edge importance.
    top = region_df.head(top_n).iloc[::-1]
    cmap = cm.get_cmap("YlOrRd")
    norm = mcolors.Normalize(vmin=top["importance"].min(),
                              vmax=top["importance"].max())
    colors = [cmap(norm(v)) for v in top["importance"]]

    fig, ax = plt.subplots(figsize=(9, 0.45 * top_n + 1.5))
    bars = ax.barh(top["region"], top["importance"],
                   color=colors, edgecolor="white", height=0.7)
    ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=8)
    ax.set_xlabel("Normalised Importance")
    ax.set_title(f"Top {top_n} Brain Regions — GNN Edge Importance")
    ax.set_xlim(0, top["importance"].max() * 1.15)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Region bar chart → {save_path}")


def plot_importance_std(all_masks, edge_df, top_n=30, save_path=IMPORTANCE_STD_PATH):
    # Bar chart of mean ± 1 SD importance for the top-N edges,
    # showing cross-subject variability in which connections are informative.
    mean_ = all_masks.mean(axis=0)
    std_ = all_masks.std(axis=0)
    top_idx = np.argsort(mean_)[::-1][:top_n]

    labels = []
    for idx in top_idx:
        if idx < len(edge_df):
            row = edge_df.iloc[idx]
            labels.append(
                f"{str(row['source_region'])[:12]}→{str(row['target_region'])[:12]}"
            )
        else:
            labels.append(str(idx))

    x = np.arange(top_n)
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(x, mean_[top_idx], color="steelblue", alpha=0.75,
           width=0.7, label="Mean importance")
    ax.errorbar(x, mean_[top_idx], yerr=std_[top_idx],
                fmt="none", color="black", capsize=3, lw=1.5, label="±1 SD")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=70, ha="right", fontsize=7)
    ax.set_ylabel("Edge Importance")
    ax.set_title(f"Top {top_n} Edges — Mean ± SD across Subjects")
    ax.legend(fontsize=9)
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Importance ±SD chart → {save_path}")


# Run Explainer Pipeline

NUM_TOP_EDGES = 100
MAX_SUBJECTS = None  # Set to an integer to explain a subset of subjects
EXPLAIN_STEPS = 400
EXPLAIN_LR = 0.03
SPARSITY_COEF = 0.02

print("Population-level edge importance explanation")

mean_mask, union_src, union_tgt, edge_w_mean, all_masks = explain_population(
    frozen_model = model,
    dataset = val_ds,
    subject_list = list(val_subj),
    scaler = scaler,
    max_subjects = MAX_SUBJECTS,
    steps = EXPLAIN_STEPS,
    lr = EXPLAIN_LR,
    sparsity_coef = SPARSITY_COEF,
)

num_regions = len(region_names)
edge_df, region_df = build_importance_tables(
    mean_mask, union_src, union_tgt, edge_w_mean, region_names
)

print("\nTop 15 most important edges")
print(edge_df[["source_region", "target_region", "weight", "importance"]]
      .head(15).to_string(index=False))

print("\nTop 15 most important regions")
print(region_df.head(15).to_string(index=False))

print("\nTop 15 least important regions")
print(region_df.tail(15).to_string(index=False))

conn_matrix = build_conn_matrix(edge_df, num_regions, num_top=NUM_TOP_EDGES)

print("\Making 3-D connectome")
plot_3d_connectome_inline(
    conn_matrix, coords,
    title=f"GNN Edge Importance — top {NUM_TOP_EDGES} edges",
    edge_thresh="85%", node_size=6, linewidth=3,
)

region_scores_arr = (
    region_df.set_index("region")
             .reindex(region_names)["importance"]
             .fillna(0.0)
             .values
)
plot_2d_glass_brain(conn_matrix, coords, region_scores_arr)
plot_region_bar(region_df, top_n=20)
plot_importance_std(all_masks, edge_df, top_n=30)

print("\nAll outputs written to /content/")

# Save full region ranking to CSV
region_df["rank"] = region_df.index + 1
region_df[["rank", "region", "importance"]].to_csv(REGION_RANKINGS_PATH, index=False)
print(f"Region rankings → {REGION_RANKINGS_PATH}")

Loaded 406 regions
Coords shape: (406, 3)

First 5 regions:
  [0] L_V1                  MNI: [ 9.81 84.54 -0.71]
  [1] L_MST                 MNI: [42.47 66.19 10.14]
  [2] L_V6                  MNI: [14.69 80.24 31.43]
  [3] L_V2                  MNI: [12.33 80.79  2.54]
  [4] L_V3                  MNI: [17.94 84.27  4.87]

Atlas matches model: 406 regions
Population-level edge importance explanation


NameError: name 'scaler' is not defined